# Notebook 3: HMM 서사 구조 모델

## 목표
CNN 예측 역할 시퀀스를 HMM으로 학습. log-likelihood 기반 이상 덱 탐지.

## 예상 소요 시간
- 학습: 10~20분

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

BASE_DIR = '/content/drive/MyDrive/dadeum_ml'
LABELS_DIR = f'{BASE_DIR}/labels'
MODELS_DIR = f'{BASE_DIR}/models'

In [ ]:
!pip install -q hmmlearn scipy
print('설치 완료')

## 0. CNN 예측 기반 시퀀스 생성

HMM은 weak_label이 아닌 CNN pred_label로 학습 — 학습·추론 분포 일치를 위해.

In [ ]:
import pandas as pd
import numpy as np
import ast, json
from pathlib import Path
from tqdm import tqdm

ROLE_NAMES = ['표지', '섹션헤더', '본문', '도표/시각자료', '마무리']
NUM_ROLES = 5

pred_csv = Path(f'{LABELS_DIR}/labeled_with_preds.csv')

if pred_csv.exists():
    pred_df   = pd.read_csv(pred_csv)
    label_col = 'pred_label'
    print('CNN 예측 기반 시퀀스 사용 (labeled_with_preds.csv)')
else:
    pred_df   = pd.read_csv(f'{LABELS_DIR}/weak_labels.csv')
    label_col = 'weak_label'
    print('⚠ CNN 예측 없음 — weak_label 사용 (step2 먼저 실행 권장)')

sequences_cnn = []
for deck_id, group in pred_df.groupby('deck_id'):
    group = group.sort_values('slide_idx')
    seq   = group[label_col].tolist()
    if len(seq) < 2:
        continue
    sequences_cnn.append({'deck_id': deck_id, 'sequence': seq, 'length': len(seq)})

seq_df = pd.DataFrame(sequences_cnn)
seq_df.to_csv(f'{LABELS_DIR}/sequences_cnn.csv', index=False)
print(f'CNN 기반 시퀀스: {len(seq_df)}개')

## 1. 시퀀스 데이터 로드

In [ ]:
# sequences_cnn.csv의 sequence 컬럼은 "[0, 2, 4]" 형태 문자열 — 파싱
seq_df['sequence'] = seq_df['sequence'].apply(ast.literal_eval)

print(f'시퀀스 수: {len(seq_df)}')
print(f'평균 길이: {seq_df["length"].mean():.1f}장')

import matplotlib.pyplot as plt
seq_df['length'].hist(bins=30, figsize=(8, 4))
plt.xlabel('슬라이드 수')
plt.ylabel('덱 수')
plt.title('덱 길이 분포')
plt.show()

## 2. 전이 행렬 시각화

In [ ]:
import seaborn as sns

transition_counts = np.zeros((NUM_ROLES, NUM_ROLES), dtype=int)

for seq in seq_df['sequence']:
    for i in range(len(seq) - 1):
        transition_counts[seq[i]][seq[i+1]] += 1

transition_prob = transition_counts / (transition_counts.sum(axis=1, keepdims=True) + 1e-8)

plt.figure(figsize=(8, 6))
sns.heatmap(
    transition_prob,
    annot=True, fmt='.2f',
    xticklabels=ROLE_NAMES,
    yticklabels=ROLE_NAMES,
    cmap='YlOrRd'
)
plt.xlabel('다음 슬라이드 역할')
plt.ylabel('현재 슬라이드 역할')
plt.title('역할 전이 확률 행렬 (CNN pred 기반)')
plt.tight_layout()
plt.savefig(f'{MODELS_DIR}/transition_matrix.png', dpi=100)
plt.show()
print('전이 확률 행렬 저장 완료')

## 3. HMM 학습 — BIC 기반 n_components 선택

탐색 범위 [3,4,5,6,7], n_iter=200, BIC 최솟값 기준 선택.

In [ ]:
from hmmlearn import hmm
import pickle
from sklearn.model_selection import train_test_split

def prepare_hmm_data(sequences):
    X = np.concatenate([np.array(s).reshape(-1, 1) for s in sequences])
    lengths = [len(s) for s in sequences]
    return X, lengths


train_seqs, val_seqs = train_test_split(
    seq_df['sequence'].tolist(), test_size=0.15, random_state=42
)

X_train, lengths_train = prepare_hmm_data(train_seqs)
X_val, lengths_val = prepare_hmm_data(val_seqs)

print(f'학습 시퀀스: {len(train_seqs)}개')
print(f'검증 시퀀스: {len(val_seqs)}개')


def compute_hmm_bic(model, X, lengths) -> tuple:
    """HMM BIC 계산. 낮을수록 좋다."""
    ll        = model.score(X, lengths)
    n_samples = len(X)
    n_obs     = 5   # NUM_ROLES
    nc        = model.n_components
    n_params  = (nc - 1) + nc * (nc - 1) + nc * (n_obs - 1)
    bic       = -2 * ll + n_params * np.log(n_samples)
    return bic, ll


results = []
for nc in [3, 4, 5, 6, 7]:
    m = hmm.CategoricalHMM(n_components=nc, n_iter=200, random_state=42, verbose=False)
    m.fit(X_train, lengths_train)
    bic, ll = compute_hmm_bic(m, X_val, lengths_val)
    results.append({'n_components': nc, 'bic': bic, 'val_score': ll / len(X_val), 'model': m})
    print(f'n_components={nc} | BIC={bic:.1f} | val ll/step={ll/len(X_val):.4f}')

best_result = min(results, key=lambda r: r['bic'])
best_model  = best_result['model']
best_n      = best_result['n_components']
print(f'\n최적 은닉 상태 수: {best_n} (BIC={best_result["bic"]:.1f})')

In [ ]:
# BIC 커브 시각화
bic_values = [r['bic'] for r in results]
nc_values  = [r['n_components'] for r in results]

plt.figure(figsize=(7, 4))
plt.plot(nc_values, bic_values, 'o-', color='steelblue')
plt.axvline(best_n, color='red', linestyle='--', label=f'최적 n={best_n}')
plt.xlabel('n_components')
plt.ylabel('BIC (낮을수록 좋음)')
plt.title('HMM BIC 커브')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f'{MODELS_DIR}/hmm_bic_curve.png', dpi=100)
plt.show()

## 4. 임계값 결정 — 정규성 검정 후 방법 선택

In [ ]:
# 전체 덱의 log-likelihood 계산
all_scores = []
for seq in seq_df['sequence'].tolist():
    x = np.array(seq).reshape(-1, 1)
    score = best_model.score(x) / len(seq)
    all_scores.append(score)

scores_np = np.array(all_scores)

from scipy import stats

stat, p_value = stats.shapiro(scores_np[:5000])
is_normal     = p_value > 0.05
print(f'Shapiro-Wilk: stat={stat:.4f}, p={p_value:.4f}')

if is_normal:
    threshold_method  = 'gaussian_3sigma'
    threshold_primary = float(np.mean(scores_np) - 3 * np.std(scores_np))
else:
    threshold_method  = 'percentile'
    threshold_primary = float(np.percentile(scores_np, 5))

thresholds = {
    'method':            threshold_method,
    'threshold_primary': threshold_primary,
    'threshold_5pct':    float(np.percentile(scores_np, 5)),
    'threshold_10pct':   float(np.percentile(scores_np, 10)),
    'mean':              float(np.mean(scores_np)),
    'std':               float(np.std(scores_np)),
    'shapiro_p':         float(p_value),
    'n_decks':           len(scores_np),
}

# 분포 시각화
plt.figure(figsize=(10, 4))
plt.hist(scores_np, bins=50, alpha=0.7, color='steelblue', edgecolor='white')
plt.xlabel('Log-Likelihood per Step')
plt.ylabel('덱 수')
plt.title(f'정상 덱 Log-Likelihood 분포 (method={threshold_method})')
plt.axvline(thresholds['threshold_5pct'], color='red', linestyle='--',
            label=f'5th pct: {thresholds["threshold_5pct"]:.3f}')
plt.axvline(thresholds['threshold_10pct'], color='orange', linestyle='--',
            label=f'10th pct: {thresholds["threshold_10pct"]:.3f}')
plt.axvline(threshold_primary, color='purple', linestyle='-',
            label=f'primary ({threshold_method}): {threshold_primary:.3f}')
plt.legend()
plt.tight_layout()
plt.savefig(f'{MODELS_DIR}/hmm_score_distribution.png', dpi=100)
plt.show()

## 4-B. 모델 및 임계값 저장

In [ ]:
with open(f'{MODELS_DIR}/hmm_model.pkl', 'wb') as f:
    pickle.dump(best_model, f)
print(f'HMM 모델 저장: {MODELS_DIR}/hmm_model.pkl (n_components={best_n})')

with open(f'{MODELS_DIR}/hmm_thresholds.json', 'w') as f:
    json.dump(thresholds, f, indent=2)
print(f'임계값 저장: {MODELS_DIR}/hmm_thresholds.json')
print(json.dumps(thresholds, indent=2))

## 5. HMM 해석 — 학습된 전이 행렬 확인

In [ ]:
emission_df = pd.DataFrame(
    best_model.emissionprob_,
    columns=ROLE_NAMES,
    index=[f'은닉상태 {i}' for i in range(best_n)]
)
print('방출 확률 (은닉 상태 → 관측 역할):')
print(emission_df.round(3))

state_names = []
for i in range(best_n):
    dominant_role = ROLE_NAMES[emission_df.iloc[i].argmax()]
    state_names.append(f'State{i}({dominant_role})')

plt.figure(figsize=(6, 5))
sns.heatmap(
    best_model.transmat_,
    annot=True, fmt='.2f',
    xticklabels=state_names,
    yticklabels=state_names,
    cmap='Blues'
)
plt.title('HMM 은닉 상태 전이 확률')
plt.tight_layout()
plt.savefig(f'{MODELS_DIR}/hmm_transition.png', dpi=100)
plt.show()

## 6. 길이 편향 검증

In [ ]:
lengths_arr = np.array([len(seq) for seq in seq_df['sequence'].tolist()])
corr, p_corr = stats.pearsonr(lengths_arr, all_scores)
print(f'길이-점수 피어슨 상관: r={corr:.3f}, p={p_corr:.4f}')
if abs(corr) > 0.3:
    print('⚠ 길이 편향 존재. 스코어링 방식 재검토 필요.')
else:
    print('✓ 길이 편향 없음 (per-step 정규화 효과 확인)')

plt.figure(figsize=(7, 4))
plt.scatter(lengths_arr, all_scores, alpha=0.3, s=5)
plt.xlabel('덱 슬라이드 수')
plt.ylabel('Log-Likelihood per Step')
plt.title(f'길이-점수 상관 (r={corr:.3f})')
plt.tight_layout()
plt.savefig(f'{MODELS_DIR}/hmm_length_bias_check.png', dpi=100)
plt.show()

## 7. 이상 시퀀스 샘플 출력

In [ ]:
score_series = pd.Series(all_scores, index=seq_df['deck_id'])
anomaly_ids  = score_series[score_series < thresholds['threshold_primary']].index.tolist()

print(f'\n이상 탐지된 덱: {len(anomaly_ids)}개 (상위 10개 출력)')
for deck_id in anomaly_ids[:10]:
    seq      = seq_df[seq_df['deck_id'] == deck_id].iloc[0]['sequence']
    readable = ' → '.join([ROLE_NAMES[r] for r in seq])
    print(f'  {deck_id} (score={score_series[deck_id]:.3f}): {readable}')

In [ ]:
print('=== Notebook 3 완료 ===')
print(f'HMM 모델: {MODELS_DIR}/hmm_model.pkl')
print(f'이상 임계값: {MODELS_DIR}/hmm_thresholds.json')
print(f'최적 은닉 상태 수: {best_n}')
print(f'임계값 방법: {threshold_method}')
print(f'sequences_cnn.csv: {LABELS_DIR}/sequences_cnn.csv')
print('\nNotebook 4 (최종 평가)로 이동하세요.')